# P1. Data project: propensity score matching on the Lalonde data (MatchIt version)

Did the National Supported Work job-training program (`treat`) raise 1978 earnings (`re78`)? The confounders
are age, education, race, marital status, high school degree, and 1974 and 1975 earnings.

Two packages ship a dataset called `lalonde`. This notebook uses `MatchIt`'s, where race is a single factor
(`black` / `hispan` / `white`). `p2-lalonde-matching.ipynb` uses `Matching`'s version, with separate `black` and
`hisp` indicators. The course used the older `black` + `hispan` columns; with the `race` factor the propensity
score model is the same model, coded differently.

In [1]:
source("setup.R")
use_packages("tableone", "Matching", "MatchIt")

data("lalonde", package = "MatchIt")
head(lalonde)

xvars <- c("age", "educ", "race", "married", "nodegree", "re74", "re75")

,treat,age,educ,race,married,nodegree,re74,re75,re78
,<int>,<int>,<int>,<fct>,<int>,<int>,<dbl>,<dbl>,<dbl>
NSW1,1,37,11,black,1,1,0,0,9930.0460
NSW2,1,22,9,hispan,0,1,0,0,3595.8940
NSW3,1,30,12,black,0,0,0,0,24909.4500
NSW4,1,27,11,black,0,1,0,0,7506.1460
NSW5,1,33,8,black,0,1,0,0,289.7899
NSW6,1,22,9,black,0,1,0,0,4056.4940


## Before matching

In [2]:
print(CreateTableOne(vars = xvars, strata = "treat", data = lalonde, test = FALSE), smd = TRUE)

# raw (unadjusted) difference in mean 1978 earnings, treated minus untreated
mean(lalonde$re78[lalonde$treat == 1]) - mean(lalonde$re78[lalonde$treat == 0])

                      Stratified by treat
                       0                 1                 SMD   
  n                        429               185                 
  age (mean (SD))        28.03 (10.79)     25.82 (7.16)     0.242
  educ (mean (SD))       10.24 (2.86)      10.35 (2.01)     0.045
  race (%)                                                  1.701
     black                  87 (20.3)        156 (84.3)          
     hispan                 61 (14.2)         11 ( 5.9)          
     white                 281 (65.5)         18 ( 9.7)          
  married (mean (SD))     0.51 (0.50)       0.19 (0.39)     0.719
  nodegree (mean (SD))    0.60 (0.49)       0.71 (0.46)     0.235
  re74 (mean (SD))     5619.24 (6788.75) 2095.57 (4886.62)  0.596
  re75 (mean (SD))     2466.48 (3292.00) 1532.06 (3219.25)  0.287


[1] -635.0262

## Propensity score

Logistic regression of treatment on the confounders only (not on the outcome `re78`).

In [3]:
psmodel <- glm(treat ~ age + educ + race + married + nodegree + re74 + re75,
               family = binomial(link = "logit"), data = lalonde)
summary(psmodel)
pscore <- psmodel$fitted.values
range(pscore)


Call:
glm(formula = treat ~ age + educ + race + married + nodegree + 
    re74 + re75, family = binomial(link = "logit"), data = lalonde)

Coefficients:
              Estimate Std. Error z value Pr(>|z|)    
(Intercept) -1.663e+00  9.709e-01  -1.713  0.08668 .  
age          1.578e-02  1.358e-02   1.162  0.24521    
educ         1.613e-01  6.513e-02   2.477  0.01325 *  
racehispan  -2.082e+00  3.672e-01  -5.669 1.44e-08 ***
racewhite   -3.065e+00  2.865e-01 -10.699  < 2e-16 ***
married     -8.321e-01  2.903e-01  -2.866  0.00415 ** 
nodegree     7.073e-01  3.377e-01   2.095  0.03620 *  
re74        -7.178e-05  2.875e-05  -2.497  0.01253 *  
re75         5.345e-05  4.635e-05   1.153  0.24884    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 751.49  on 613  degrees of freedom
Residual deviance: 487.84  on 605  degrees of freedom
AIC: 505.84

Number of Fisher Scoring iterations: 5


[1] 0.009080193 0.853152844

## Matching on the propensity score

Pair matching without replacement on the propensity score itself (not its logit), first without a caliper.
The seed makes the matched set reproducible.

In [4]:
# standardized mean difference, as tableone computes it: (mean_t - mean_c) / sqrt((var_t + var_c) / 2)
smd <- function(data, col, group = "treat") {
  xt <- data[data[[group]] == 1, col]
  xc <- data[data[[group]] == 0, col]
  (mean(xt) - mean(xc)) / sqrt((var(xt) + var(xc)) / 2)
}

set.seed(931139)
psmatch <- Match(Tr = lalonde$treat, M = 1, X = pscore, replace = FALSE)
matched <- matched_rows(psmatch, lalonde)
print(CreateTableOne(vars = xvars, strata = "treat", data = matched, test = FALSE), smd = TRUE)
c(smd_married = smd(matched, "married"))

                      Stratified by treat
                       0                 1                 SMD   
  n                        185               185                 
  age (mean (SD))        25.29 (10.65)     25.82 (7.16)     0.058
  educ (mean (SD))       10.55 (2.71)      10.35 (2.01)     0.084
  race (%)                                                  0.854
     black                  87 (47.0)        156 (84.3)          
     hispan                 39 (21.1)         11 ( 5.9)          
     white                  59 (31.9)         18 ( 9.7)          
  married (mean (SD))     0.20 (0.40)       0.19 (0.39)     0.027
  nodegree (mean (SD))    0.65 (0.48)       0.71 (0.46)     0.127
  re74 (mean (SD))     2351.12 (4192.62) 2095.57 (4886.62)  0.056
  re75 (mean (SD))     1605.02 (2601.68) 1532.06 (3219.25)  0.025


smd_married 
-0.02723636

## With a caliper of 0.1

In [5]:
set.seed(931139)
psmatch <- Match(Tr = lalonde$treat, M = 1, X = pscore, replace = FALSE, caliper = 0.1)
matched <- matched_rows(psmatch, lalonde)
c(matched_pairs = length(psmatch$index.treated))
print(CreateTableOne(vars = xvars, strata = "treat", data = matched, test = FALSE), smd = TRUE)

matched_pairs 
          111

                      Stratified by treat
                       0                 1                 SMD   
  n                        111               111                 
  age (mean (SD))        26.27 (11.10)     26.22 (7.18)     0.006
  educ (mean (SD))       10.37 (2.66)      10.25 (2.31)     0.047
  race (%)                                                  0.041
     black                  80 (72.1)         82 (73.9)          
     hispan                 12 (10.8)         11 ( 9.9)          
     white                  19 (17.1)         18 (16.2)          
  married (mean (SD))     0.24 (0.43)       0.24 (0.43)    <0.001
  nodegree (mean (SD))    0.66 (0.48)       0.65 (0.48)     0.019
  re74 (mean (SD))     2704.56 (4759.89) 2250.49 (5746.14)  0.086
  re75 (mean (SD))     1969.10 (3169.08) 1222.25 (3081.19)  0.239


## Outcome analysis on the matched pairs

In [6]:
y_trt <- matched$re78[matched$treat == 1]
y_con <- matched$re78[matched$treat == 0]
diffy <- y_trt - y_con
mean(diffy)
t.test(diffy)

[1] 1246.806


	One Sample t-test

data:  diffy
t = 1.4824, df = 110, p-value = 0.1411
alternative hypothesis: true mean is not equal to 0
95 percent confidence interval:
 -420.0273 2913.6398
sample estimates:
mean of x 
 1246.806 
